In [ ]:
import torch, numpy as np, math, time
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEV)


In [ ]:
# ── Task: cue accumulation ──
N_IN = 40; N_OUT = 2; N_CUES = 7
T_CUE = 15; T_GAP = 5; T_INTERVAL = T_CUE + T_GAP
T_RECALL = 25
F_HI = 0.25; F_LO = 0.05; NC = N_IN // 4

def seq_len(t_wait):
    """Total trial length for a given delay (T_WAIT)."""
    return N_CUES * T_INTERVAL + t_wait + T_RECALL

def gen_batch(B, t_wait):
    T = seq_len(t_wait)
    x = (torch.rand(B, T, N_IN, device=DEV) < F_LO).float()
    sides = (torch.rand(B, N_CUES, device=DEV) < 0.5).long()
    for k in range(N_CUES):
        t0 = k * T_INTERVAL; c = sides[:, k]
        for cc in (0, 1):
            m = (c == cc)
            if m.any():
                x[m, t0:t0+T_CUE, cc*NC:(cc+1)*NC] = (
                    torch.rand(int(m.sum()), T_CUE, NC, device=DEV) < F_HI
                ).float()
    x[:, -T_RECALL:, 2*NC:3*NC] = (torch.rand(B, T_RECALL, NC, device=DEV) < F_HI).float()
    label = (sides.sum(1) > N_CUES // 2).long()
    return x, label

print(f'Default trial length (T_WAIT=50): {seq_len(50)} timesteps')


In [ ]:
# ── Network parameters ───────────────────────────────────────────────────
N_REC = 100
ALPHA = 0.005   # leak rate

def init(seed):
    g = torch.Generator(device='cpu').manual_seed(seed)
    W_in  = (torch.randn(N_REC, N_IN,  generator=g) / math.sqrt(N_IN )).to(DEV)
    W_rec = (torch.randn(N_REC, N_REC, generator=g) / math.sqrt(N_REC)).to(DEV)
    W_out = (torch.randn(N_OUT, N_REC, generator=g) / math.sqrt(N_REC)).to(DEV)
    b_out = torch.zeros(N_OUT, device=DEV)
    return {'W_in': W_in, 'W_rec': W_rec, 'W_out': W_out, 'b_out': b_out}

def cos(a, b):
    a = a.flatten(); b = b.flatten()
    return (a @ b / (a.norm() * b.norm() + 1e-12)).item()


In [ ]:
def eprop_tanh(P, x, label, t_wait):
    """Symmetric e-prop for a leaky tanh-RNN (single layer)."""
    T = seq_len(t_wait)
    B = x.shape[0]
    W_rec_diag = torch.diag(P['W_rec'])

    h       = torch.zeros(B, N_REC, device=DEV)
    eps_in  = torch.zeros(B, N_REC, N_IN,  device=DEV)
    eps_rec = torch.zeros(B, N_REC, N_REC, device=DEV)
    E_in    = torch.zeros(B, N_REC, N_IN,  device=DEV)
    E_rec   = torch.zeros(B, N_REC, N_REC, device=DEV)
    h_sum   = torch.zeros(B, N_REC, device=DEV)

    for t in range(T):
        xt     = x[:, t]
        h_prev = h
        pre    = h @ P['W_rec'].t() + xt @ P['W_in'].t()
        h      = (1 - ALPHA) * h + ALPHA * torch.tanh(pre)
        d_h    = ALPHA * (1 - torch.tanh(pre) ** 2)
        local_jac = (1 - ALPHA) + d_h * W_rec_diag.unsqueeze(0)

        eps_in  = local_jac.unsqueeze(2) * eps_in  + d_h.unsqueeze(2) * xt.unsqueeze(1)
        eps_rec = local_jac.unsqueeze(2) * eps_rec + d_h.unsqueeze(2) * h_prev.unsqueeze(1)

        if t >= T - T_RECALL:
            E_in  += eps_in
            E_rec += eps_rec
            h_sum += h

    logits = (h_sum / T_RECALL) @ P['W_out'].t() + P['b_out']
    p   = torch.softmax(logits, 1)
    oh  = torch.zeros(B, N_OUT, device=DEV); oh[torch.arange(B), label] = 1.
    err = p - oh

    L = (err @ P['W_out']) / T_RECALL

    gIn  = (L.unsqueeze(2) * E_in ).sum(0) / B
    gRec = (L.unsqueeze(2) * E_rec).sum(0) / B
    gOut = (err.t() @ (h_sum / T_RECALL)) / B
    gb   = err.mean(0)

    acc  = (logits.argmax(1) == label).float().mean().item()
    loss = torch.nn.functional.cross_entropy(logits, label).item()
    return {'W_in': gIn, 'W_rec': gRec, 'W_out': gOut, 'b_out': gb}, loss, acc


def loss_bptt_tanh(P, x, label, t_wait):
    """Exact BPTT (autograd through the unrolled tanh-RNN) -- ground truth."""
    T = seq_len(t_wait)
    B = x.shape[0]
    h = torch.zeros(B, N_REC, device=DEV)
    h_sum = torch.zeros(B, N_REC, device=DEV)
    for t in range(T):
        pre = h @ P['W_rec'].t() + x[:, t] @ P['W_in'].t()
        h   = (1 - ALPHA) * h + ALPHA * torch.tanh(pre)
        if t >= T - T_RECALL:
            h_sum = h_sum + h
    logits = (h_sum / T_RECALL) @ P['W_out'].t() + P['b_out']
    loss = torch.nn.functional.cross_entropy(logits, label)
    acc = (logits.argmax(1) == label).float().mean().item()
    return loss, acc


In [ ]:
# ── Quick gradient sanity check at init ──────────────────────────────────
P = init(0)
x, label = gen_batch(64, t_wait=50)
g_eprop, _, _ = eprop_tanh(P, x, label, 50)
Pg = {k: P[k].clone().requires_grad_(True) for k in P}
loss_bptt_tanh(Pg, x, label, 50)[0].backward()

print('Gradient cosine vs BPTT (untrained network, T_WAIT=50):')
for k in P:
    print(f'  {k:6s}  single-layer e-prop cos = {cos(g_eprop[k], Pg[k].grad):+.4f}')


In [ ]:
def train_eprop(seed, t_wait, iters, batch, eval_every):
    torch.manual_seed(seed)
    P = init(seed)
    for k in P: P[k].requires_grad_(True)
    opt = torch.optim.Adam([
        {'params': [P['W_in'], P['W_rec']], 'lr': 1e-3},
        {'params': [P['W_out'], P['b_out']], 'lr': 3e-3}
    ])
    steps, accs = [], []
    for it in range(iters):
        x, label = gen_batch(batch, t_wait)
        g, loss, acc = eprop_tanh(P, x, label, t_wait)
        opt.zero_grad()
        for k in P: P[k].grad = g[k]
        opt.step()
        if it % eval_every == 0 or it == iters - 1:
            with torch.no_grad():
                xv, lv = gen_batch(200, t_wait)
                _, _, av = eprop_tanh(P, xv, lv, t_wait)
            steps.append(it); accs.append(av)
    return steps, accs


def train_bptt(seed, t_wait, iters, batch, eval_every):
    torch.manual_seed(seed)
    P = init(seed)
    for k in P: P[k].requires_grad_(True)
    opt = torch.optim.Adam([
        {'params': [P['W_in'], P['W_rec']], 'lr': 1e-3},
        {'params': [P['W_out'], P['b_out']], 'lr': 3e-3}
    ])
    steps, accs = [], []
    for it in range(iters):
        x, label = gen_batch(batch, t_wait)
        opt.zero_grad()
        loss, acc = loss_bptt_tanh(P, x, label, t_wait)
        loss.backward()
        opt.step()
        if it % eval_every == 0 or it == iters - 1:
            with torch.no_grad():
                xv, lv = gen_batch(200, t_wait)
                _, av = loss_bptt_tanh(P, xv, lv, t_wait)
            steps.append(it); accs.append(av)
    return steps, accs


def steps_to_threshold(steps, accs, thresh=0.90):
    """First training step at which held-out accuracy crosses `thresh`.
    If never reached, returns the final step as a (censored) lower bound."""
    for s, a in zip(steps, accs):
        if a >= thresh:
            return s, True
    return steps[-1], False


In [ ]:
# ── Configuration (tune for your Colab runtime) ──────────────────────────
N_SEEDS    = 8
ITERS      = 1000
EVAL_EVERY = 20
BATCH      = 48
T_WAIT_DEFAULT = 50

t0 = time.time()
results = {'BPTT': [], 'single-layer e-prop': []}

for s in range(N_SEEDS):
    results['BPTT'].append(
        train_bptt(s, T_WAIT_DEFAULT, ITERS, BATCH, EVAL_EVERY))
    results['single-layer e-prop'].append(
        train_eprop(s, T_WAIT_DEFAULT, ITERS, BATCH, EVAL_EVERY))
    print(f'seed {s} done ({time.time()-t0:.0f}s elapsed)')

print(f'\nTotal training time: {time.time()-t0:.0f}s')


In [ ]:
# ── Plot 1: learning curves (left) + steps-to-90% bar chart (right) ─────
colors = {'BPTT': 'dimgray', 'single-layer e-prop': 'tab:blue'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# -- left: learning curves with shaded 95% CI across seeds --
ax = axes[0]
for name, runs in results.items():
    steps = runs[0][0]
    acc_mat = np.array([r[1] for r in runs])      # (N_SEEDS, n_evals)
    mean = acc_mat.mean(0)
    sem  = acc_mat.std(0, ddof=1) / np.sqrt(len(runs)) if len(runs) > 1 else np.zeros_like(mean)
    ci95 = 1.96 * sem
    ax.plot(steps, mean, marker='o', ms=3, color=colors[name], label=name)
    ax.fill_between(steps, mean - ci95, mean + ci95, color=colors[name], alpha=0.2)
ax.axhline(0.5, ls='--', color='gray', label='chance')
ax.set_ylim(0.45, 1.02)
ax.set_xlabel('training step'); ax.set_ylabel('accuracy (held-out)')
ax.set_title(f'Cue accumulation (T_WAIT={T_WAIT_DEFAULT}), {N_SEEDS} seeds')
ax.legend(fontsize=8)

# -- right: steps to reach acc >= 0.90 --
ax = axes[1]
names = list(results.keys())
means, errs, censored = [], [], []
for name in names:
    vals = []
    any_censored = False
    for steps, accs in results[name]:
        s, reached = steps_to_threshold(steps, accs, 0.90)
        vals.append(s)
        any_censored = any_censored or (not reached)
    vals = np.array(vals)
    means.append(vals.mean())
    errs.append(vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0)
    censored.append(any_censored)

bars = ax.bar(names, means, yerr=errs, capsize=4,
               color=[colors[n] for n in names])
for i, c in enumerate(censored):
    if c:
        ax.text(i, means[i] + errs[i] + max(means)*0.03, '>= (not reached\nby some seeds)',
                 ha='center', fontsize=7, color='gray')
ax.set_ylabel('training steps to reach acc >= 0.90')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=15, ha='right')

plt.tight_layout()
plt.savefig('plot1_learning_curves_and_speed.png', dpi=150)
plt.show()


In [ ]:
# ── Statistical test: is the BPTT-vs-e-prop steps-to-90% gap real? ──────
# The bar chart above shows *mean +/- SEM* per method. SEM bars are narrower
# than a 95% CI and don't by themselves say whether the two methods differ
# reliably. Here we test it properly on the per-seed values.
from scipy import stats

steps90 = {}
for name in results:
    vals = []
    for steps, accs in results[name]:
        s90, reached = steps_to_threshold(steps, accs, 0.90)
        vals.append(s90)
    steps90[name] = np.array(vals, dtype=float)

bptt_vals  = steps90['BPTT']
eprop_vals = steps90['single-layer e-prop']

def mean_ci95(vals):
    n = len(vals)
    m = vals.mean()
    sem = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0.0
    tcrit = stats.t.ppf(0.975, df=n-1) if n > 1 else 0.0
    return m, sem, tcrit * sem

bptt_m, bptt_sem, bptt_ci95   = mean_ci95(bptt_vals)
eprop_m, eprop_sem, eprop_ci95 = mean_ci95(eprop_vals)

# Welch's t-test (unequal variances) -- primary test
t_stat, t_p = stats.ttest_ind(bptt_vals, eprop_vals, equal_var=False)
# Mann-Whitney U -- nonparametric cross-check (robust to the small, discrete,
# right-skewed nature of "steps to threshold" data)
u_stat, u_p = stats.mannwhitneyu(bptt_vals, eprop_vals, alternative='two-sided')
# Effect size
pooled_sd = np.sqrt((bptt_vals.std(ddof=1)**2 + eprop_vals.std(ddof=1)**2) / 2)
cohens_d = (bptt_m - eprop_m) / pooled_sd if pooled_sd > 0 else float('nan')

print(f"BPTT:  mean={bptt_m:.1f} steps, 95% CI = [{bptt_m-bptt_ci95:.1f}, {bptt_m+bptt_ci95:.1f}]  (n={len(bptt_vals)})")
print(f"e-prop: mean={eprop_m:.1f} steps, 95% CI = [{eprop_m-eprop_ci95:.1f}, {eprop_m+eprop_ci95:.1f}]  (n={len(eprop_vals)})")
print(f"95% CIs overlap: {(bptt_m-bptt_ci95 <= eprop_m+eprop_ci95) and (eprop_m-eprop_ci95 <= bptt_m+bptt_ci95)}")
print()
print(f"Welch's t-test:      t={t_stat:.3f}, p={t_p:.3f}")
print(f"Mann-Whitney U test: U={u_stat:.1f}, p={u_p:.3f}")
print(f"Cohen's d (effect size): {cohens_d:.3f}")
print()
alpha = 0.05
if t_p < alpha and u_p < alpha:
    print(f"=> Significant at alpha={alpha}: the steps-to-90% gap is unlikely to be pure seed noise.")
else:
    print(f"=> NOT significant at alpha={alpha}: with n={N_SEEDS} seeds, the observed gap is "
          f"consistent with seed-to-seed noise -- can't reliably claim one method learns faster.")


In [ ]:
def delay_sweep_point(t_wait, n_avg=8, batch=64):
    cos_full, cos_out = [], []
    for s in range(n_avg):
        P = init(1000 + s)
        x, label = gen_batch(batch, t_wait)
        g_eprop, _, _ = eprop_tanh(P, x, label, t_wait)
        Pg = {k: P[k].clone().requires_grad_(True) for k in P}
        loss_bptt_tanh(Pg, x, label, t_wait)[0].backward()

        full_vec = torch.cat([g_eprop['W_in'].flatten(), g_eprop['W_rec'].flatten()])
        bptt_vec = torch.cat([Pg['W_in'].grad.flatten(), Pg['W_rec'].grad.flatten()])

        cos_full.append(cos(full_vec, bptt_vec))
        cos_out.append(cos(g_eprop['W_out'], Pg['W_out'].grad))

    def m_ci(vals):
        vals = np.array(vals)
        mean = vals.mean()
        sem  = vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
        return mean, 1.96 * sem

    return (m_ci(cos_full), m_ci(cos_out))


DELAYS = [10, 25, 50, 100, 200]   # T_WAIT values (timesteps) to sweep
N_AVG  = 8                        # random seeds/batches averaged per delay point

sweep = [delay_sweep_point(d, n_avg=N_AVG) for d in DELAYS]
print('delay sweep done')


In [ ]:
# ── Plot 2: gradient cosine vs delay ─────────────────────────────────────
cos_full_m = [s[0][0] for s in sweep]; cos_full_e = [s[0][1] for s in sweep]
cos_out_m  = [s[1][0] for s in sweep]; cos_out_e  = [s[1][1] for s in sweep]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.errorbar(DELAYS, cos_out_m,  yerr=cos_out_e,  marker='o', mfc='white',
            ls=':',  color='tab:blue',  label='single-layer e-prop · output-adjacent (W_out)')
ax.errorbar(DELAYS, cos_full_m, yerr=cos_full_e, marker='o',
            ls='-',  color='tab:blue',  label='single-layer e-prop · input-adjacent (W_in+W_rec)')
ax.axhline(0, ls=':', color='gray')
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('delay D (timesteps)'); ax.set_ylabel('cosine vs BPTT')
ax.set_title('Gradient cosine similarity vs. delay')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plot2_cosine_vs_delay.png', dpi=150)
plt.show()
